## 데이터 로드 및 전처리

### 문제 1-1
- ../data/ml-latest-small/movies.csv, ../data/ml-latest-small/ratings.csv 파일을 읽고 각각의 shape를 출력하시오.

In [35]:
import pandas as pd
import numpy as np

In [36]:
movies_df = pd.read_csv('../data/ml-latest-small/movies.csv')
ratings_df = pd.read_csv('../data/ml-latest-small/ratings.csv')

print(movies_df.shape, ratings_df.shape)

movies_df.head()

(9742, 3) (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### 문제 1-2
- ratings_df와 movies_df를 movieId 기준으로 병합하여 movie_rating_df를 생성하시오.

In [37]:
movie_rating_df = pd.merge(ratings_df, movies_df, on='movieId', how='inner')
print(movie_rating_df.shape)
movie_rating_df.head()

(100836, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


### 문제 1-3
- pivot_table()을 사용하여 사용자-영화 평점 행렬 user_movie_df를 생성하시오.
    - index: userId
    - columns: title
    - values: rating
    - 평점이 없는 값은 0으로 채우시오.

In [38]:
user_movie_df = movie_rating_df.pivot_table('rating', index='userId', columns='title', fill_value=0)    # NaN을 0으로 채움
print(user_movie_df.shape)
user_movie_df.head()

(610, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 사용자 기반 협업 필터링 준비

### 문제 2-1
- cosine_similarity()를 사용하여 사용자-사용자 유사도 행렬을 계산하시오. 결과를 user_sim_df DataFrame으로 저장하시오.
- (힌트: 사용자 기반 협업 필터링에서는 사용자-영화 행렬을 그대로 사용한다.)

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

user_sim = cosine_similarity(user_movie_df, user_movie_df)
print(user_sim.shape)

# 유저 id를 index, column에 모두 지정
user_sim_df = pd.DataFrame(user_sim, index=user_movie_df.T.columns, columns=user_movie_df.T.columns)
user_sim_df.head()

(610, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


### 문제 2-2
- 실제 평점이 있는 위치에서만 MSE를 계산하는 movie_rating_mse(actual, pred) 함수를 작성하시오.

In [40]:
from sklearn.metrics import mean_squared_error

def predict_ratings_by_all_user(user_movie_df,user_sim_df):
    return (user_movie_df.T @ user_sim_df) / np.abs(user_sim_df).sum(axis=1)
rating_pred_df = predict_ratings_by_all_user(user_movie_df,user_sim_df)

def user_rating_mse(actual,pred):
    non_zero_idx = actual.nonzero()
    actual = actual[non_zero_idx]
    pred = pred[non_zero_idx]
    return mean_squared_error(actual,pred)

user_rating_mse(user_movie_df.T.values,rating_pred_df.values)

9.444684826235058

### 문제 2-3
- 특정 사용자가 보지 않은 영화 목록을 반환하는 get_unseen_movies(user_idx) 함수를 작성하시오.

In [41]:
def get_unseen_movies(user_idx):
    user_ratings = user_movie_df.T.iloc[user_idx]
    return user_ratings[user_ratings == 0]

## 사용자 기반 예측 평점 계산

### 문제 3-1
- 상위 유사 사용자 topn명을 이용해 모든 사용자의 모든 영화에 대한 예측 평점을 계산하는 predict_ratings_user_based(topn=20) 함수를 작성하시오.

- 다음 흐름을 따르시오.
    1. 각 사용자마다 유사도가 높은 다른 사용자를 찾는다.
    2. 자기 자신은 제외한다.
    3. 상위 topn명의 평점만 사용한다.
    4. 실제로 평점을 준 영화만 반영한다.
    5. 유사도를 가중치로 사용하여 예측 평점을 계산한다.
    6. 분모가 0이면 0으로 처리한다.

In [ ]:
def predict_ratings(topn=20):
    ratings = user_movie_df.values  # (num_users, num_movies)
    sim = user_sim_df.values       # (num_movies, num_movies)

    num_users, num_movies = ratings.shape
    pred = np.zeros((num_users,num_movies))     # 예측 결과로 사용할 배열

    for movie_idx in range(num_users):
        topn_sim_idx = np.argsort(sim[user_idx])[::-1]

        # 자기 자신 제외 후 topn 선택
        topn_sim_idx = topn_sim_idx[topn_sim_idx != user_idx][:topn]

        # 현재 영화와 topn 유사 영화들의 유사도
        topn_sim = sim[movie_idx, topn_sim_idx]

        # 모든 사용자의 topn 유사 영화들의 평점
        topn_ratings = ratings[:,topn_sim_idx]

        # 사용자가 실제로 본 영화만 반영
        non_zero_mask = topn_ratings != 0

        # 분자 : 유사도 * 평점의 합
        numer = (topn_ratings * topn_sim * non_zero_mask).sum(axis=1)

        # 분모 : 실제로 본 유사 영화들에 해당하는 유사도 절대값 합
        denom = (np.abs(topn_sim)*non_zero_mask).sum(axis=1)

        # 0으로 나누기 방지
        pred[:,movie_idx] = np.divide(
            numer,
            denom,
            out=np.zeros(num_users),
            where=denom != 0
        )
    return pred

ratings_pred = predict_ratings()
print(ratings_pred.shape)

IndexError: index 610 is out of bounds for axis 0 with size 610

### 문제 3-2
- 예측 결과를 rating_pred_user_df DataFrame으로 변환하시오.

### 문제 3-3
- 사용자 기반 협업 필터링의 예측 MSE를 출력하시오.

## 사용자 기반 영화 추천

### 문제 4-1
- 특정 사용자가 보지 않은 영화 중 예측 평점이 높은 영화를 추천하는 recommend_movies_user_based(user_idx, topn=20) 함수를 작성하시오.

- 반환 형식은 아래 컬럼을 갖는 DataFrame으로 하시오.
    - title
    - pred_rating
    - user_rating

### 문제 4-2
- 예시로 recommend_movies_user_based(100)의 결과를 확인하시오.

## 해석
- 아이템 기반 협업 필터링과 사용자 기반 협업 필터링의 차이를 한두 문장으로 정리하시오.